# 04.6 Garbage Collection and Reference Counting

You never call `free()` in Python. Memory is reclaimed automatically — but
"automatically" hides two distinct mechanisms, and knowing both explains when
cleanup happens and when it does not.

## Theory

### Mechanism 1: reference counting

Every CPython object carries a counter of how many references point at it.

```
data = [1, 2]        # refcount 1
alias = data         # refcount 2
del alias            # refcount 1
del data             # refcount 0  -> freed immediately
```

When the count hits zero, the memory is released **at once** — deterministically,
at that exact line. This handles the overwhelming majority of objects.

**What changes the count:**

| Increases | Decreases |
|---|---|
| Assigning to a name | `del name` |
| Adding to a container | Removing from a container |
| Passing to a function | Function returns |
| Setting as an attribute | Rebinding the name |

### Mechanism 2: the cycle collector

Reference counting has one blind spot: **cycles**.

```python
a = {}
b = {}
a["points_to"] = b
b["points_to"] = a
del a, b             # both refcounts are still 1 - neither is freed
```

Each object is kept alive by the other. Nothing outside can reach them, but
neither count reaches zero.

The **generational garbage collector** (`gc` module) finds these unreachable
cycles and frees them. It runs periodically, not immediately — so cycle cleanup
is **not** deterministic.

### Generations

The collector sorts objects into three generations. New objects start in
generation 0, which is scanned most often. Objects surviving a scan are promoted.
The rationale — the **generational hypothesis** — is that most objects die young,
so scanning new objects frequently and old ones rarely is the best use of effort.

### Why this matters practically

- Relying on `__del__` firing at a specific moment is **unsafe**
- Files and sockets should be closed by `with`, not by garbage collection
- Cycles are handled, but later — and not at all if you disable the collector

In [ ]:
import sys

# getrefcount always reports one extra, for the temporary argument reference.
def refcount(obj):
    """Return the real reference count, excluding the temporary."""
    return sys.getrefcount(obj) - 1


# Watch the count move.
data = [1, 2, 3]
print("after data = [1, 2, 3]      ", refcount(data))

alias = data
print("after alias = data          ", refcount(data))

container = [data, data]
print("after putting it in a list twice", refcount(data))

holder = {"key": data}
print("after adding to a dict      ", refcount(data))

del alias
print("after del alias             ", refcount(data))

del container
print("after del container         ", refcount(data))

del holder
print("after del holder            ", refcount(data))

print("")
print("When the count would reach 0, CPython frees the object immediately.")

## Seeing deallocation happen

A class with `__del__` lets us observe the exact moment of collection.

In [ ]:
class Tracked:
    """Announces its own destruction."""

    def __init__(self, label):
        self.label = label
        print(f"   created: {self.label}")

    def __del__(self):
        # Called when the object is about to be destroyed.
        print(f"   destroyed: {self.label}")


print("Creating an object with one reference:")
item = Tracked("first")

print("")
print("Rebinding the name - the old object has no references left:")
item = Tracked("second")

print("")
print("Deleting the name:")
del item

print("")
print("Notice the destruction happened at an exact, predictable line.")
print("That is reference counting - not the cycle collector.")

In [ ]:
# With two references, deletion of one is not enough.
first_name = Tracked("shared")
second_name = first_name

print("")
print("Two names point at it. Deleting one:")
del first_name
print("   (nothing destroyed - one reference remains)")

print("")
print("Deleting the second:")
del second_name

## The blind spot: reference cycles

Two objects referring to each other keep each other alive, even when nothing else
can reach them.

In [ ]:
import gc

class Node:
    """A node that can point at another node."""

    def __init__(self, label):
        self.label = label
        self.partner = None

    def __del__(self):
        print(f"   destroyed: {self.label}")


# Stop the automatic collector so we can control the timing.
gc.disable()

# Build a cycle: each node references the other.
left = Node("left")
right = Node("right")
left.partner = right
right.partner = left

print("Cycle built. Deleting both names:")
del left, right
print("   (nothing was destroyed - the cycle keeps both alive)")

print("")
print("Running the collector manually:")
collected = gc.collect()
print(f"   gc.collect() freed {collected} objects")

# Re-enable normal behaviour.
gc.enable()

That is the whole problem in one cell. Each node's refcount stayed at 1 because
its partner still pointed at it. Only the cycle collector could see that the pair
was unreachable as a group.

In normal operation the collector runs on its own, so you would not need
`gc.collect()`. The demonstration disabled it purely to make the timing visible.

## Inspecting the collector

In [ ]:
import gc

print("Collector enabled?", gc.isenabled())

# Thresholds control when each generation is scanned.
threshold_zero, threshold_one, threshold_two = gc.get_threshold()
print("")
print("Thresholds:")
print("   generation 0 scanned after", threshold_zero,
      "more allocations than deallocations")
print("   generation 1 scanned after", threshold_one, "generation-0 scans")
print("   generation 2 scanned after", threshold_two, "generation-1 scans")

# Current object counts per generation.
counts = gc.get_count()
print("")
print("Objects currently tracked per generation:", counts)

# Total tracked objects.
print("Total objects the collector is tracking:", len(gc.get_objects()))

print("")
print("Generation 0 holds new objects and is scanned most often.")
print("Survivors are promoted, on the theory that most objects die young.")

## Why `__del__` is unreliable

`__del__` runs when an object is destroyed — but you cannot control *when* that
is, and in some cases it may not run at all.

In [ ]:
class Resource:
    """Pretends to hold a file handle."""

    def __init__(self, name):
        self.name = name
        self.open = True

    def close(self):
        self.open = False
        print(f"   closed: {self.name}")

    def __del__(self):
        # Relying on this is the problem.
        if self.open:
            print(f"   __del__ closing: {self.name}")
            self.open = False


print("Relying on __del__:")
resource = Resource("data.txt")
del resource
print("   worked here - but only because the refcount hit zero")

print("")
print("Cases where __del__ timing is NOT guaranteed:")
problems = [
    "the object is part of a reference cycle",
    "the interpreter is shutting down",
    "an exception traceback is still holding a reference",
    "you are running PyPy or another implementation",
    "the object is referenced from a closure or a cache",
]
for problem in problems:
    print("   -", problem)

print("")
print("THE FIX: use a context manager. Chapter 29 covers this properly.")

class SafeResource:
    """Closes deterministically, via the with statement."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"   opened: {self.name}")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        # Guaranteed to run, even if an exception occurred.
        print(f"   closed: {self.name}")


print("")
print("With a context manager:")
with SafeResource("data.txt"):
    print("   ...using the resource...")

## An exception can keep objects alive

A traceback holds references to every frame in the call stack — which keeps their
local variables alive longer than you might expect.

In [ ]:
import sys

class Large:
    """Stands in for an object holding a lot of memory."""

    def __init__(self):
        self.payload = list(range(1000))


def failing_function():
    """Create an object, then raise."""
    big = Large()
    raise ValueError("something went wrong")


# Catching the exception keeps the traceback - and the frame - alive.
try:
    failing_function()
except ValueError as error:
    # error.__traceback__ references the frame where `big` lived.
    frame = error.__traceback__.tb_next.tb_frame
    print("The traceback still holds the failed frame's locals:")
    print("   local names:", list(frame.f_locals))
    print("   'big' is still reachable:", "big" in frame.f_locals)

print("")
print("This is why `except X as e:` deletes `e` at the end of the block -")
print("Python does it automatically to break that reference.")

print("")
print("After the except block, is the name still bound?")
try:
    error
except NameError:
    print("   no - Python deleted it to release the traceback")

## Memory is not always returned to the OS

CPython manages small objects in its own pools. Freeing an object usually returns
memory to Python's allocator, not to the operating system — so process memory
often stays flat rather than dropping.

In [ ]:
import sys

# Object sizes, to give a sense of scale.
samples = [
    ("empty list", []),
    ("list of 1000 ints", list(range(1000))),
    ("empty dict", {}),
    ("small string", "hello"),
    ("empty tuple", ()),
    ("empty set", set()),
]

print("Object                    Bytes")
print("-" * 38)
for label, value in samples:
    print(label.ljust(25), sys.getsizeof(value))

print("")
print("Note getsizeof is SHALLOW - it does not count referenced objects:")

inner = list(range(1000))
outer = [inner, inner, inner]

print("   list of 1000 ints:", sys.getsizeof(inner), "bytes")
print("   list holding it 3 times:", sys.getsizeof(outer),
      "bytes <- only the 3 references")

print("")
print("Practical points:")
print("   - freeing objects rarely shrinks the process's memory footprint")
print("   - CPython reuses freed blocks from its own pools")
print("   - measure with tracemalloc, not by watching the OS. Chapter 37.")

## Takeaways

1. CPython frees objects with **reference counting** — when the count hits zero,
   the memory is released **immediately** at that line.
2. Names, containers, arguments and attributes all hold references.
   `sys.getrefcount(x) - 1` shows the real count.
3. Reference counting cannot free **cycles**, because each object keeps the other
   alive.
4. The **generational garbage collector** finds unreachable cycles and frees
   them, but runs periodically — so cycle cleanup is not deterministic.
5. Three generations exist because **most objects die young**; new objects are
   scanned most often.
6. **Never rely on `__del__`** for cleanup — use `with` and a context manager.
7. A caught exception's traceback keeps frames alive, which is why Python deletes
   the `as` name at the end of an `except` block.
8. Freeing objects does not usually return memory to the operating system.

## Try it yourself

1. Track a list's reference count as you add it to containers and delete them.
2. Write a class with `__del__` and find the exact line where it fires.
3. Build a two-object cycle with `gc.disable()`. Confirm nothing is freed until
   `gc.collect()`.
4. Compare `sys.getsizeof()` of a list of 1000 integers with a list holding the
   same list three times. Why the difference?
5. Find a place in your own code relying on `__del__`. Rewrite it with `with`.